## **Multi-Head Attention**

In [5]:
import torch 
import math
import torch.nn as nn
import torch.nn.functional as F

## **Input Embeddings**

In [6]:
##initialize the vocab size and its model dimension

class InputEmbeddings(nn.Module):
    ##initialize 
    def __init__(self, vocab_size: int, d_model:int)->None:
        super().__init__()
        # Set the model dimensionality and vocabulary size
        self.d_model = d_model
        self.vocab_size = vocab_size
        #forming Embedding Matrix with shape = (vocab_size, d_model)
        # Instantiate the embedding layer
        self.embeddings = nn.Embedding(vocab_size, d_model)
    
    ##forword feed
    def forward(self, x):
        # Return the embeddings multiplied by the square root of d_model
        ##scaled up factors to scale embeddings
        scaled_factor = math.sqrt(self.d_model)
        #dense vector representation
        tensor = self.embeddings(x)
        return tensor*scaled_factor

## **Positional Encoding**

In [7]:
##positional encoding layers
##define the positional encoding class by inherting from torch module
class PositionalEncoding(nn.Module):
    ##intialize
    def __init__(self, d_model:int, max_seq_length:int):
        super().__init__()
        # Create a matrix of zeros of dimensions max_seq_length by d_model
        ##initialize the positional embeddings (pe) to zeros
        pe = torch.zeros(max_seq_length, d_model)
        ##create the tensor of positions for each token in the sequence then it transform using unsqueeze
        ##so that it can be used in positional encoding calculations. unsqueeze(1)- converts into tensor.
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        ##division term in sin and cosine function
        div_term = torch.exp(torch.arange(0,d_model, 2, dtype=torch.float)* -(math.log(10000.0)/d_model))

        ##sin and cosine calculation
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        ##store pe without making learnable parameter during training
        self.register_buffer('pe', pe.unsqueeze(0))

    ##adding input token embeddings (X_input_token) with positional embedding (pe)
    def forward(self, x_embed_token):
        ##operation: element-wise addition
        #x_input_token + positional_encoding
        #token_representation + position_representation
        return x_embed_token+self.pe[:, :x_embed_token.size(1)]

## **Multi-Head Attention**

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads): ##num_heads is number of attention head, each handling embeddings of size head_dim
        super().__init__()
        ##applying constraint on head_dim using (assert) method
        assert d_model%num_heads==0, "d_model must be divisible by num heads"
        ##initializing the d_model and num_heads
        # Calculate the dimensions each head will process
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model//num_heads

        ##initialize three embedding matrices of equal dimension
        ##Three linear layers are defined for the attention inputs
        ##bias = False, no impact on performance while reducing complexity (only for inputs): Q,K,V layers eliminates the bias term,
        ##reducing model parameters without impacting the ability to capture relationships.
        # Define the three input layers and one output layer
        self.query_linear = nn.Linear(d_model, d_model, bias=False) 
        self.key_linear = nn.Linear(d_model, d_model, bias=False)
        self.values_linear = nn.Linear(d_model, d_model, bias=False)
        ##final concatenated output layer
        self.output_linear = nn.Linear(d_model, d_model)
    
    ##Three different helper method
    ##split and transform the input embeddings between the attention heads
    def split_heads(self, x, batch_size):
        seq_length = x.size(1)
        # Split the input embeddings and permute
        x=x.reshape(batch_size, seq_length, self.num_heads, self.head_dim)
        ##rearrange the position of vector/element using index
        return x.permute(0,2,1,3)
    
    ##compute attention weights using F.softmax
    # Compute scaled dot-product attention
    def compute_attention(self, query, key, value, mask=None):
        ##matrix multiplication using matmul method of torch
        ##requires to transpose Key-matrix, and calculate the attention weights inside each head using softmax
        scores = torch.matmul(query, key.transpose(-2, -1))/(self.head_dim**0.5)
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))
        ##calculating attention weights
        attention_weights = F.softmax(scores, dim=-1)
        ##return these attention weights matrix multiplied by the value matrix
        return torch.matmul(attention_weights, value)

    ##transform the attention weights back into the original embedding shape
    def combine_heads(self,x, batch_size):
        x = x.permute(0,2,1,3).contiguous()
        # Combine heads back to (batch_size, seq_length, d_model)
        return x.view(batch_size, -1, self.d_model)
    
    ##forward method 
    def forward(self, query, key, value, mask =None):
        batch_size = query.size(0)

        Query = self.split_heads(self.query_linear(query), batch_size)
        Key = self.split_heads(self.key_linear(key), batch_size)
        Values = self.split_heads(self.values_linear(value), batch_size)

        ##attention weights
        attention_weights = self.compute_attention(Query, Key, Values, mask)

        ##output
        output = self.combine_heads(attention_weights, batch_size)
        return self.output_linear(output)




In [1]:
d_model = 512
num_heads = 8

d_model//num_heads

64

In [ ]:
# Define attention parameters
d_model = 512
num_heads = 8

# Instantiate a MultiHeadAttention instance
multihead_attn = MultiHeadAttention(d_model,num_heads)

# Pass the query, key, and value matrices through the mechanism
output = multihead_attn(query, key, value)
print(output.shape)